# Physics Diagnostic Notebook

This notebook verifies the MATLAB gradient/noise convention and compares exact analytical coefficient recovery with the saved learned coefficient head. It does not train a model. Run the cells manually in Jupyter or VS Code.

In [1]:
from pathlib import Path
import json
import sys
import numpy as np
import pandas as pd
import torch

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / 'src').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))

from models.one_model import MultiTaskGradientModel

N = 10
SIGMA = 0.01
DATA_ROOT = ROOT / 'outputs' / 'final_one_stage_sigma01_optimization'
MULTITASK_ROOT = DATA_ROOT / 'multitask_coordinate' / 'N_10'
NOTES_ROOT = ROOT / 'docs' / 'experiments' / 'sigma01_results'
NOTES_ROOT.mkdir(parents=True, exist_ok=True)
print('Root:', ROOT)
print('Dataset root:', MULTITASK_ROOT / 'datasets')

Root: c:\Users\prest\Documents\minimal-gravimetry-ml
Dataset root: c:\Users\prest\Documents\minimal-gravimetry-ml\outputs\final_one_stage_sigma01_optimization\multitask_coordinate\N_10\datasets


## Load Data

The training split contains noisy gradients. Validation and test gradients are clean, matching the MATLAB/task protocol confirmed by the advisor.

In [2]:
dataset_root = MULTITASK_ROOT / 'datasets'
train = np.load(dataset_root / 'train.npz')
validation = np.load(dataset_root / 'validation.npz')
test = np.load(dataset_root / 'test.npz')

num_coefficients = N + 1
num_measurements = N + 1

def features_to_complex(features):
    midpoint = features.shape[-1] // 2
    return features[..., :midpoint] + 1j * features[..., midpoint:]

def coefficients_to_features(values):
    return np.concatenate([values.real, values.imag], axis=-1).astype(np.float32)

train_coefficients = features_to_complex(train['coefficients'])
validation_coefficients = features_to_complex(validation['coefficients'])
test_coefficients = features_to_complex(test['coefficients'])
train_gradients = features_to_complex(train['gradient_data'])
validation_gradients = features_to_complex(validation['gradient_data'])
test_gradients = features_to_complex(test['gradient_data'])

print('Train:', train_gradients.shape, 'Validation:', validation_gradients.shape, 'Test:', test_gradients.shape)

Train: (40000, 11) Validation: (2000, 11) Test: (1000, 11)


## MATLAB Forward-Equation Check

MATLAB defines each complex gradient as `sum(2 * conj(A_n) * x^(n+1))`. This cell reproduces that equation and checks it against the saved clean validation/test gradients.

In [3]:
measurement_points = np.exp(1j * np.linspace(0.0, 2.0 * np.pi, num_measurements, endpoint=False))
orders = np.arange(num_coefficients)
gradient_matrix = 2.0 * measurement_points[:, None] ** (orders[None, :] + 1)

def matlab_gradients(coefficients):
    return np.einsum('mn,bn->bm', gradient_matrix, np.conj(coefficients))

reconstructed_validation_gradients = matlab_gradients(validation_coefficients)
reconstructed_test_gradients = matlab_gradients(test_coefficients)
forward_check = pd.DataFrame([{
    'validation_max_abs_error': float(np.max(np.abs(reconstructed_validation_gradients - validation_gradients))),
    'test_max_abs_error': float(np.max(np.abs(reconstructed_test_gradients - test_gradients))),
    'measurement_matrix_condition_number': float(np.linalg.cond(gradient_matrix)),
}])
forward_check

,validation_max_abs_error,test_max_abs_error,measurement_matrix_condition_number
0,3.156335e-08,3.816883e-08,1.0


## Empirical Noise Check

The saved training shapes also contain their clean coefficients, so we can regenerate their clean gradients and measure the actual injected noise.

In [4]:
clean_train_gradients = matlab_gradients(train_coefficients)
noise = train_gradients - clean_train_gradients
noise_check = pd.DataFrame([{
    'configured_sigma': SIGMA,
    'real_mean': float(np.mean(noise.real)),
    'real_std': float(np.std(noise.real)),
    'imag_mean': float(np.mean(noise.imag)),
    'imag_std': float(np.std(noise.imag)),
    'complex_abs_mean': float(np.mean(np.abs(noise))),
    'complex_abs_std': float(np.std(np.abs(noise))),
}])
noise_check

,configured_sigma,real_mean,real_std,imag_mean,imag_std,complex_abs_mean,complex_abs_std
0,0.01,0.000019,0.010005,-0.000006,0.009996,0.012533,0.006554


## Analytical Coefficient Recovery

Because the gradient equation uses `conj(A_n)`, the pseudoinverse first recovers the conjugated coefficients and then conjugates them back. This is a diagnostic baseline, not a trained model.

In [5]:
gradient_pseudoinverse = np.linalg.pinv(gradient_matrix)

def recover_coefficients(gradients):
    conjugated_coefficients = np.einsum('nm,bm->bn', gradient_pseudoinverse, gradients)
    return np.conj(conjugated_coefficients)

analytical_clean_validation = recover_coefficients(validation_gradients)
analytical_clean_test = recover_coefficients(test_gradients)
analytical_noisy_train = recover_coefficients(train_gradients)

analytical_check = pd.DataFrame([{
    'clean_validation_mae': float(np.mean(np.abs(analytical_clean_validation - validation_coefficients))),
    'clean_test_mae': float(np.mean(np.abs(analytical_clean_test - test_coefficients))),
    'sigma01_noisy_train_mae': float(np.mean(np.abs(analytical_noisy_train - train_coefficients))),
    'clean_validation_max_abs_error': float(np.max(np.abs(analytical_clean_validation - validation_coefficients))),
}])
analytical_check

,clean_validation_mae,clean_test_mae,sigma01_noisy_train_mae,clean_validation_max_abs_error
0,3.217560e-10,3.152573e-10,0.001889,7.475083e-09


## Learned Coefficient-Head Comparison

This loads the saved multitask one-stage checkpoint. Its coefficient head is the available learned Stage 1 proxy. No training occurs here.

In [6]:
model_path = MULTITASK_ROOT / 'multitask_model_weights.pt'
learned_check = None
if model_path.exists():
    model = MultiTaskGradientModel(
        input_dim=2 * num_measurements, coefficient_dim=2 * num_coefficients,
        output_dim=32 * 32, hidden_dims=(512, 1024), dropout_rates=(0.1, 0.1),
        latent_grid_size=16, latent_channels=160, decoder_channels=(160, 128, 96, 64, 32),
    )
    model.load_state_dict(torch.load(model_path, map_location='cpu', weights_only=True))
    model.eval()
    input_mean = train['gradient_data'].mean(axis=0)
    input_std = np.where(train['gradient_data'].std(axis=0) == 0, 1.0, train['gradient_data'].std(axis=0))
    target_mean = train['coefficients'].mean(axis=0)
    target_std = np.where(train['coefficients'].std(axis=0) == 0, 1.0, train['coefficients'].std(axis=0))
    def learned_coefficients(features):
        normalized = torch.tensor((features - input_mean) / input_std, dtype=torch.float32)
        with torch.no_grad():
            predictions = model(normalized)[1].cpu().numpy()
        return features_to_complex(predictions * target_std + target_mean)
    learned_noisy_train = learned_coefficients(train['gradient_data'])
    learned_clean_validation = learned_coefficients(validation['gradient_data'])
    learned_clean_test = learned_coefficients(test['gradient_data'])
    learned_check = pd.DataFrame([{
        'analytical_sigma01_noisy_train_mae': float(np.mean(np.abs(analytical_noisy_train - train_coefficients))),
        'learned_sigma01_noisy_train_mae': float(np.mean(np.abs(learned_noisy_train - train_coefficients))),
        'analytical_clean_validation_mae': float(np.mean(np.abs(analytical_clean_validation - validation_coefficients))),
        'learned_clean_validation_mae': float(np.mean(np.abs(learned_clean_validation - validation_coefficients))),
        'analytical_clean_test_mae': float(np.mean(np.abs(analytical_clean_test - test_coefficients))),
        'learned_clean_test_mae': float(np.mean(np.abs(learned_clean_test - test_coefficients))),
    }])
else:
    print('Saved multitask checkpoint not found:', model_path)
learned_check

,analytical_sigma01_noisy_train_mae,learned_sigma01_noisy_train_mae,analytical_clean_validation_mae,learned_clean_validation_mae,analytical_clean_test_mae,learned_clean_test_mae
0,0.001889,0.00122,3.217560e-10,0.000892,3.152573e-10,0.000894


In [7]:
diagnostic = {
    'N': N, 'sigma': SIGMA,
    'forward_check': forward_check.to_dict(orient='records'),
    'noise_check': noise_check.to_dict(orient='records'),
    'analytical_check': analytical_check.to_dict(orient='records'),
    'learned_check': None if learned_check is None else learned_check.to_dict(orient='records'),
}
(NOTES_ROOT / 'physics_diagnostic_sigma01.json').write_text(json.dumps(diagnostic, indent=2), encoding='utf-8')
forward_check.to_csv(NOTES_ROOT / 'physics_forward_check.csv', index=False)
noise_check.to_csv(NOTES_ROOT / 'physics_noise_check.csv', index=False)
analytical_check.to_csv(NOTES_ROOT / 'physics_analytical_recovery.csv', index=False)
if learned_check is not None:
    learned_check.to_csv(NOTES_ROOT / 'physics_learned_comparison.csv', index=False)
print('Diagnostic notes written to', NOTES_ROOT)

Diagnostic notes written to c:\Users\prest\Documents\minimal-gravimetry-ml\docs\experiments\sigma01_results
